# CO544: Machine Learning and Data Mining
### Lab 05: Text Classification and Performance Analysis

#### E/21/245 -  MADHUSHAN S.K.A.K.

#### Text Classification

####(a) Importing required modules

In [1]:
import re  # Regular expressions
from sklearn.datasets import load_files  # to load the dataset
import nltk  # Natural Language Toolkit

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

#### (b) Loading data

In [2]:
from sklearn.datasets import load_files
from nltk.stem import WordNetLemmatizer


In [3]:
#Instantiate lemmatizer (needed for later)
lemmatizer = WordNetLemmatizer()


In [4]:
# Change the path to point directly inside the inner folder
path_to_dataset = "/content/drive/MyDrive/movie reviews/movie_reviews"

# Load the dataset again
movie_data = load_files(path_to_dataset)
X, y = movie_data.data, movie_data.target



In [5]:
# Show basic summary information
print(f"Number of documents: {len(X)}")
print(f"Number of labels: {len(y)}")
print(f"Target names (classes): {movie_data.target_names}")

#  Decode and print a preview of the first review
print("\nFirst document (decoded):")
print(X[0].decode('utf-8')[:500])  # Show first 500 characters

Number of documents: 2000
Number of labels: 2000
Target names (classes): ['neg', 'pos']

First document (decoded):
arnold schwarzenegger has been an icon for action enthusiasts , since the late 80's , but lately his films have been very sloppy and the one-liners are getting worse . 
it's hard seeing arnold as mr . freeze in batman and robin , especially when he says tons of ice jokes , but hey he got 15 million , what's it matter to him ? 
once again arnold has signed to do another expensive blockbuster , that can't compare with the likes of the terminator series , true lies and even eraser . 
in this so cal


#### (c) Data preprocessing


In [6]:
documents = []

for i in range(len(X)):
    # 1. Decode from bytes to string
    document = X[i].decode('utf-8')

    # 2. Apply your regex substitutions
    document = re.sub(r'\W', ' ', document)          # Remove special characters
    document = re.sub(r'^[a-zA-Z]\s+', ' ', document) # Remove single chars at beginning
    document = re.sub(r'\s+[a-zA-Z]\s+', ' ', document) # Remove single chars in middle
    document = re.sub(r'\d+', '', document)           # Remove numbers
    document = re.sub(r'\s+', ' ', document, flags=re.I) # Replace multiple spaces with a single space

    # 3. Lowercase
    document = document.lower()

    # 4. Tokenize
    document = document.split()

    # 5. Lemmatize
    document = [lemmatizer.lemmatize(word) for word in document]

    # 6. Rejoin tokens if needed (optional)
    document = ' '.join(document)

    # 7. Append to new list
    documents.append(document)

# Print a preview of the first preprocessed document to verify
print("Preprocessing completed! Here is a preview of the first document:")
print(documents[0][:500])

Preprocessing completed! Here is a preview of the first document:
arnold schwarzenegger ha been an icon for action enthusiast since the late but lately his film have been very sloppy and the one liner are getting worse it hard seeing arnold a mr freeze in batman and robin especially when he say ton of ice joke but hey he got million what it matter to him once again arnold ha signed to do another expensive blockbuster that can compare with the like of the terminator series true lie and even eraser in this so called dark thriller the devil gabriel byrne ha come 


#### (d) Convert text into numbers


In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords

# Initialize CountVectorizer with constraints
vectorizer = CountVectorizer(
    max_features=1500,
    min_df=7,
    max_df=0.8,
    stop_words=stopwords.words('english')
)


In [8]:

# Convert the text documents into numerical vectors
X_vectors = vectorizer.fit_transform(documents).toarray()

# To check the shape and vocabulary
print(f"Shape of X_vectors: {X_vectors.shape}")
print("\nList of feature words (First 20 words):")
print(vectorizer.get_feature_names_out()[:20])

Shape of X_vectors: (2000, 1500)

List of feature words (First 20 words):
['ability' 'able' 'absolutely' 'academy' 'accent' 'accident' 'across'
 'act' 'acting' 'action' 'actor' 'actress' 'actual' 'actually' 'ad' 'adam'
 'adaptation' 'add' 'added' 'addition']


#### (e) Text Classification

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vectors, y,
    test_size=0.2,
    random_state=0
)


In [12]:
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)



LogisticRegression()

In [13]:

predictions = log_reg.predict(X_test)

#### (f) Evaluating Model Performance

In [14]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Confusion Matrix:
[[164  44]
 [ 28 164]]


In [15]:
print("\nClassification Report:")
print(classification_report(y_test, predictions))


Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.79      0.82       208
           1       0.79      0.85      0.82       192

    accuracy                           0.82       400
   macro avg       0.82      0.82      0.82       400
weighted avg       0.82      0.82      0.82       400



In [16]:
print("\nAccuracy:")
print(accuracy_score(y_test, predictions))


Accuracy:
0.82


#### Task 3

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
import pandas as pd

In [18]:
rf_model = RandomForestClassifier(random_state=0)
svm_model = SVC(kernel='linear', random_state=0) # Using linear kernel for text data
nb_model = GaussianNB()

In [19]:
models = {
    "Logistic Regression": log_reg, # Already trained in previous step
    "Random Forest": rf_model,
    "Support Vector Machine": svm_model,
    "Naive Bayes": nb_model
}

In [20]:
results = []

In [21]:
for name, model in models.items():
    # Train the model if it's not already trained
    if name != "Logistic Regression":
        model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    # Extract precision, recall, f1-score (macro average) from classification report
    report = classification_report(y_test, y_pred, output_dict=True)
    precision = report['macro avg']['precision']
    recall = report['macro avg']['recall']
    f1 = report['macro avg']['f1-score']

    # Print Confusion Matrix individually for clarity
    print(f"=== {name} ===")
    print("Confusion Matrix:")
    print(cm)
    print("-" * 30)

    # Store metrics for final comparison table
    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4)
    })

=== Logistic Regression ===
Confusion Matrix:
[[164  44]
 [ 28 164]]
------------------------------
=== Random Forest ===
Confusion Matrix:
[[168  40]
 [ 29 163]]
------------------------------
=== Support Vector Machine ===
Confusion Matrix:
[[165  43]
 [ 33 159]]
------------------------------
=== Naive Bayes ===
Confusion Matrix:
[[165  43]
 [ 59 133]]
------------------------------


In [25]:
df_compare = pd.DataFrame(results)
print("\nFINAL PERFORMANCE COMPARISON")
df_compare


FINAL PERFORMANCE COMPARISON


,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,0.8200,0.8213,0.8213,0.8200
1,Random Forest,0.8275,0.8279,0.8283,0.8275
2,Support Vector Machine,0.8100,0.8102,0.8107,0.8100
3,Naive Bayes,0.7450,0.7461,0.7430,0.7434
